# Explore the 'unnannotated' category of branchwater matches

In [2]:
import polars as pl

## Load list of accessions with some minimum number of core species in them

In [3]:
# produced by branchwater-figure.ipynb
found_df = (
    pl.read_csv('../outputs.branchwater/branchwater_8species_unannot.csv')
    .filter(pl.col('count') >= 8)
)
min_count = found_df['count'].min()

print(f"loaded {len(found_df)} accessions with at least (>=) {min_count} of the 16 core species in them.")

loaded 2606 accessions with at least (>=) 8 of the 16 core species in them.


## Load metadata

In [3]:
# get metadata, only compare WGS
metadata_df = (
    pl.scan_parquet("/group/ctbrowngrp5/sra-metagenomes/20241128-metadata.parquet")
    .filter(pl.col("acc") != "NP")
    .filter(pl.col("assay_type") == "WGS")
#    .select(["acc", "organism", "bioproject", "mbases", "host", "project_name"]) 
    .collect()
)
print(f"found metadata records for {len(metadata_df)} WGS metagenome data sets from the SRA")

found metadata records for 703916 WGS metagenome data sets from the SRA


In [1]:
mgnify_df = pl.read_csv('../mgnify_bioproj.csv')

NameError: name 'pl' is not defined

## Join 'found' list with metadata

In [4]:
found_df2 = found_df.join(metadata_df, on='acc', how='inner', validate="1:m")
assert len(found_df) == len(found_df2)
assert found_df['count'].min() == 8

In [5]:
# for _all_ bioprojects in the metadata, do foo. CTB replace with reporting?
all_bioproject_sum_df = metadata_df.group_by('bioproject').agg(
    n_acc_total=pl.len(),
    orgs=pl.col('organism').unique(),   # list of "organism" annotations
)

In [6]:
summary_df = found_df2.group_by('bioproject').agg(
    n_acc_matched=pl.len(),
).join(all_bioproject_sum_df, on='bioproject', how='inner').sort('n_acc_matched')

summary_df.filter(pl.col('n_acc_matched') > 2).style

bioproject,n_acc_matched,n_acc_total,orgs
PRJNA635116,3,51,['gut metagenome']
PRJNA692042,3,216,['metagenome']
PRJNA944095,3,5,['metagenome']
PRJNA827108,3,4,['gut metagenome']
PRJNA677897,3,3,['metagenome']
PRJNA595610,3,3,['feces metagenome']
PRJEB15296,3,4,['metagenome']
PRJNA1116614,3,330,"['Eubacteriales', 'gut metagenome', 'Prevotella', 'Bacteroidales', 'Bifidobacteriaceae', 'bacterium', 'Lachnospiraceae', 'Methanobacteriota', 'Bacteroidota bacterium', 'Clostridia', 'Deltaproteobacteria', 'Selenomonadales', 'Veillonellaceae']"
PRJNA730942,3,11,['gut metagenome']
PRJNA825695,3,12,['metagenome']
